# Step 9: Relation-Chain Bridge Expansion Batch 2

这个 notebook 用于执行 `Round 1e` 的最后一次 `relation_chain_bridge` feasibility check。

它和 Batch 1 的区别是：

- 不再泛泛搜索单个亲属词
- 只保留更高 precision 的 multi-relation / nested-relation 模板
- 目标不是扩大候选池，而是判断当前 `HotpotQA` 是否真的能支持最小 `relation_chain_bridge` source set


In [1]:
!pip install -q datasets


In [2]:
import csv
import json
import re
from collections import Counter
from pathlib import Path

from datasets import load_dataset


/Users/mac/miniforge3/envs/mind2web/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 配置路径与批量参数


In [3]:
PROJECT_ROOT_OVERRIDE = ''
CURRENT_SPLIT = 'validation'
RAW_CANDIDATE_SIZE = 20
RESULT_DIR_NAME = '09_relation_chain_bridge_expansion_batch2'

RELATION_TERMS = [
    'father-in-law',
    'mother-in-law',
    'brother-in-law',
    'sister-in-law',
    'spouse',
    'husband',
    'wife',
    'father',
    'mother',
    'sibling',
    'brother',
    'sister',
    'son',
    'daughter',
]

HIGH_PRECISION_PATTERNS = {
    'parent_child_chain': re.compile(r'\b(mother|father)\b.{0,80}\b(son|daughter)\b|\b(son|daughter)\b.{0,80}\b(mother|father)\b', re.IGNORECASE),
    'spouse_family_chain': re.compile(r'\b(wife|husband|spouse)\b.{0,80}\b(father|mother|brother|sister|sibling|son|daughter)\b|\b(father|mother|brother|sister|sibling|son|daughter)\b.{0,80}\b(wife|husband|spouse)\b', re.IGNORECASE),
    'sibling_family_chain': re.compile(r'\b(brother|sister|sibling)\b.{0,80}\b(father|mother|wife|husband|son|daughter)\b|\b(father|mother|wife|husband|son|daughter)\b.{0,80}\b(brother|sister|sibling)\b', re.IGNORECASE),
    'nested_of_chain': re.compile(r'\b(mother|father|wife|husband|brother|sister|sibling)\b\s+to\s+the\s+\b(son|daughter)\b|\b(mother|father|wife|husband|brother|sister|sibling)\b\s+of\s+the\s+\b(son|daughter|wife|husband|brother|sister|sibling)\b', re.IGNORECASE),
    'possessive_relation_chain': re.compile(r"'s\s+(wife|husband|son|daughter|father|mother|brother|sister|sibling)", re.IGNORECASE),
}

STOP_PHRASES = [
    'father ted',
    'son of al qaeda',
    'son of al quada',
    'father of modern american shipbuilding',
]


def detect_project_root():
    candidates = []
    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/kaggle/working/2026_SelectTransfer'),
    ])

    checked = []
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(str(candidate))
        if (candidate / 'pilot' / 'archive' / 'taxonomy_round1.csv').exists() and (candidate / 'results' / '08_relation_chain_bridge_expansion' / 'batch_01_subtype_annotation_summary.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root. Checked: ' + ' | '.join(checked))


PROJECT_ROOT = detect_project_root()
ARCHIVE_DIR = PROJECT_ROOT / 'pilot' / 'archive'
RESULT_DIR = PROJECT_ROOT / 'results' / RESULT_DIR_NAME
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TAXONOMY_PATH = ARCHIVE_DIR / 'taxonomy_round1.csv'
BATCH1_SCREENING_PATH = PROJECT_ROOT / 'results' / '08_relation_chain_bridge_expansion' / 'batch_01_first_pass_screening.csv'
BATCH1_ANNOTATION_PATH = PROJECT_ROOT / 'results' / '08_relation_chain_bridge_expansion' / 'candidate_batch_for_subtype_annotation_screened.csv'

RAW_PATH = RESULT_DIR / 'candidate_batch2_raw.csv'
FILTERED_PATH = RESULT_DIR / 'candidate_batch2_filtered.csv'
ANNOTATION_PATH = RESULT_DIR / 'candidate_batch2_for_subtype_annotation.csv'
FULL_JSON_PATH = RESULT_DIR / 'candidate_batch2_full.json'
SUMMARY_PATH = RESULT_DIR / 'candidate_batch2_summary.md'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULT_DIR =', RESULT_DIR)


PROJECT_ROOT = /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer
RESULT_DIR = /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/09_relation_chain_bridge_expansion_batch2


## 2. 加载 `HotpotQA` 与 Batch 1 状态


In [4]:
hotpot = load_dataset('hotpot_qa', 'fullwiki', split=CURRENT_SPLIT)


def read_csv(path):
    with path.open() as f:
        return list(csv.DictReader(f))


taxonomy_rows = read_csv(TAXONOMY_PATH)
batch1_screening = read_csv(BATCH1_SCREENING_PATH)
batch1_annotation = read_csv(BATCH1_ANNOTATION_PATH)

batch1_seen_ids = {row['task_id'] for row in batch1_screening}
batch1_keep_ids = {row['task_id'] for row in batch1_annotation if row['keep_drop'] == 'keep'}


def task_id_from_index(idx: int) -> str:
    return f'hp_dev_{idx:04d}'


print('hotpot size =', len(hotpot))
print('batch1 seen ids =', len(batch1_seen_ids))
print('batch1 keep ids =', sorted(batch1_keep_ids))


hotpot size = 7405
batch1 seen ids = 15
batch1 keep ids = ['hp_dev_0503']


## 3. 构造高精度 relation-chain 候选池


In [5]:
RELATION_PATTERNS = {
    term: re.compile(r'(?<![A-Za-z])' + re.escape(term) + r'(?![A-Za-z])', re.IGNORECASE)
    for term in RELATION_TERMS
}


def matched_relation_terms(question: str):
    return [term for term, pattern in RELATION_PATTERNS.items() if pattern.search(question)]


def distinct_relation_terms(question: str):
    return sorted(set(matched_relation_terms(question.lower())))


def matched_template_names(question: str):
    q = question.lower()
    hits = []
    for name, pattern in HIGH_PRECISION_PATTERNS.items():
        if pattern.search(q):
            hits.append(name)
    return hits


def has_stop_phrase(question: str):
    q = question.lower()
    return any(phrase in q for phrase in STOP_PHRASES)


raw_candidates = []
for idx, example in enumerate(hotpot):
    if example.get('type') != 'bridge':
        continue

    task_id = task_id_from_index(idx)
    if task_id in batch1_seen_ids:
        continue

    question = (example.get('question') or '').strip()
    if not question:
        continue

    terms = distinct_relation_terms(question)
    templates = matched_template_names(question)
    if len(terms) < 2:
        continue
    if not templates:
        continue
    if has_stop_phrase(question):
        continue

    supporting_titles = '|'.join(example['supporting_facts']['title']) if example.get('supporting_facts') else ''
    context_titles = example.get('context', {}).get('title', []) if isinstance(example.get('context'), dict) else []
    raw_candidates.append({
        'idx': idx,
        'task_id': task_id,
        'question': question,
        'answer': example.get('answer', ''),
        'raw_type': example.get('type', ''),
        'level': example.get('level', ''),
        'matched_relation_terms': '|'.join(terms),
        'matched_chain_templates': '|'.join(templates),
        'support_titles': supporting_titles,
        'context_titles_preview': '|'.join(context_titles[:10]),
        'full_example': example,
    })

raw_candidates = sorted(
    raw_candidates,
    key=lambda row: (
        -len(row['matched_chain_templates'].split('|')),
        -len(row['matched_relation_terms'].split('|')),
        row['idx'],
    ),
)[:RAW_CANDIDATE_SIZE]

print('raw_candidates =', len(raw_candidates))
for row in raw_candidates[:10]:
    print(row['task_id'], '|', row['matched_relation_terms'], '|', row['matched_chain_templates'], '|', row['question'])


raw_candidates = 17
hp_dev_1380 | brother|wife | spouse_family_chain|sibling_family_chain|possessive_relation_chain | Abraham Lincoln Neiman and his wife Carrie Marcus Neiman cofounded Neiman Marcus with Carrie's brother Herbert in which American city in 1907?
hp_dev_7398 | brother|wife | spouse_family_chain|sibling_family_chain|nested_of_chain | Who was the brother of the wife of the Democratic Party nomination for Vice President in 1972?
hp_dev_1892 | father|son | parent_child_chain|possessive_relation_chain | Matilda of Chester, Countess of Huntingdon's father was the son of which woman?
hp_dev_2485 | husband|mother | spouse_family_chain|possessive_relation_chain | Who is the mother of Mary, Crown Princess of Denmark's husband?
hp_dev_3422 | father|mother|son | parent_child_chain | What team does the oldest son play for, from the family whose middle son plays for the Chicago Bulls and the mother and father and third son all played basketball.
hp_dev_3372 | daughter|sister | sibling_

## 4. 最小过滤与导出


In [6]:
filtered = []
for row in raw_candidates:
    if not row['support_titles']:
        continue
    filtered.append(row)


def write_csv(path, rows, fieldnames):
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row.get(k, '') for k in fieldnames})


raw_fieldnames = [
    'task_id', 'question', 'answer', 'raw_type', 'level',
    'matched_relation_terms', 'matched_chain_templates', 'support_titles', 'context_titles_preview'
]

annotation_fieldnames = [
    'task_id', 'question', 'answer', 'raw_type', 'level',
    'matched_relation_terms', 'matched_chain_templates', 'support_titles',
    'reasoning_label', 'bridge_subtype', 'keep_drop', 'note'
]

write_csv(RAW_PATH, raw_candidates, raw_fieldnames)
write_csv(FILTERED_PATH, filtered, raw_fieldnames)
write_csv(ANNOTATION_PATH, filtered, annotation_fieldnames)

json_rows = []
for row in filtered:
    json_rows.append({
        'idx': row['idx'],
        'task_id': row['task_id'],
        'question': row['question'],
        'answer': row['answer'],
        'raw_type': row['raw_type'],
        'level': row['level'],
        'matched_relation_terms': row['matched_relation_terms'],
        'matched_chain_templates': row['matched_chain_templates'],
        'support_titles': row['support_titles'],
        'context_titles_preview': row['context_titles_preview'],
        'full_example': row['full_example'],
    })

FULL_JSON_PATH.write_text(json.dumps(json_rows, indent=2, ensure_ascii=False))

term_counter = Counter()
template_counter = Counter()
for row in filtered:
    for term in row['matched_relation_terms'].split('|'):
        if term:
            term_counter[term] += 1
    for name in row['matched_chain_templates'].split('|'):
        if name:
            template_counter[name] += 1

lines = [
    '# Relation-Chain Bridge Expansion Batch 2 Summary',
    '',
    f'- split: `{CURRENT_SPLIT}`',
    f'- raw candidates: `{len(raw_candidates)}`',
    f'- filtered candidates: `{len(filtered)}`',
    '',
    '## Relation-Term Distribution',
    '',
]
for term, count in term_counter.most_common():
    lines.append(f'- `{term}`: {count}')

lines.extend(['', '## Template Distribution', ''])
for name, count in template_counter.most_common():
    lines.append(f'- `{name}`: {count}')

lines.extend([
    '',
    '## Next Step',
    '',
    '在 `candidate_batch2_for_subtype_annotation.csv` 中补：',
    '',
    '- `reasoning_label`',
    '- `bridge_subtype`',
    '- `keep_drop`',
    '- `note`',
])

SUMMARY_PATH.write_text('\n'.join(lines) + '\n')

print('Wrote:')
for path in [RAW_PATH, FILTERED_PATH, ANNOTATION_PATH, FULL_JSON_PATH, SUMMARY_PATH]:
    print('-', path)


Wrote:
- /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/09_relation_chain_bridge_expansion_batch2/candidate_batch2_raw.csv
- /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/09_relation_chain_bridge_expansion_batch2/candidate_batch2_filtered.csv
- /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/09_relation_chain_bridge_expansion_batch2/candidate_batch2_for_subtype_annotation.csv
- /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/09_relation_chain_bridge_expansion_batch2/candidate_batch2_full.json
- /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/09_relation_chain_bridge_expansion_batch2/candidate_batch2_summary.md


## 5. 批量预览


In [7]:
for row in filtered[:20]:
    print('=' * 100)
    print(row['task_id'])
    print('terms:', row['matched_relation_terms'])
    print('templates:', row['matched_chain_templates'])
    print('question:', row['question'])
    print('support_titles:', row['support_titles'])


hp_dev_1380
terms: brother|wife
templates: spouse_family_chain|sibling_family_chain|possessive_relation_chain
question: Abraham Lincoln Neiman and his wife Carrie Marcus Neiman cofounded Neiman Marcus with Carrie's brother Herbert in which American city in 1907?
support_titles: Abraham Lincoln Neiman|Abraham Lincoln Neiman|Abraham Lincoln Neiman|Abraham Lincoln Neiman|Carrie Marcus Neiman
hp_dev_7398
terms: brother|wife
templates: spouse_family_chain|sibling_family_chain|nested_of_chain
question: Who was the brother of the wife of the Democratic Party nomination for Vice President in 1972?
support_titles: Sargent Shriver|Sargent Shriver|Eunice Kennedy Shriver
hp_dev_1892
terms: father|son
templates: parent_child_chain|possessive_relation_chain
question: Matilda of Chester, Countess of Huntingdon's father was the son of which woman?
support_titles: Matilda of Chester, Countess of Huntingdon|Hugh de Kevelioc, 5th Earl of Chester
hp_dev_2485
terms: husband|mother
templates: spouse_family_